# Model Evaluation

Evaluates all 5 trained models against the test set (post 2023-08-01).

Sections:
1. Setup
2. Load models and test set
3. Brier Score
4. Calibration Curves
5. Confusion Matrix (result model)
6. Backtesting over time
7. Baseline comparison

## 1. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent.parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import (
    brier_score_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    mean_absolute_error,
)
from sklearn.calibration import calibration_curve

from src.components.features import FEATURE_COLS

MODELS_DIR = Path().resolve().parent.parent / 'models'
DATA_PATH  = Path().resolve().parent.parent / 'datasets' / 'processed' / 'feature_engineered_dataset.csv'
SPLIT_DATE = '2023-08-01'

print('Setup complete')

## 2. Load Models and Test Set

In [ ]:
# Load models
model_result = XGBClassifier()
model_result.load_model(MODELS_DIR / 'soca_result.ubj')

model_btts = XGBClassifier()
model_btts.load_model(MODELS_DIR / 'soca_btts.ubj')

model_over25 = XGBClassifier()
model_over25.load_model(MODELS_DIR / 'soca_over25.ubj')

model_over15 = XGBClassifier()
model_over15.load_model(MODELS_DIR / 'soca_over15.ubj')

model_goals = XGBRegressor()
model_goals.load_model(MODELS_DIR / 'soca_goals.ubj')

print('All 5 models loaded')

In [ ]:
# Load and split dataset
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

test = df[df['Date'] >= SPLIT_DATE].copy()

X_test = test[FEATURE_COLS]

print(f'Test set: {len(test)} matches')
print(f'Date range: {test["Date"].min().date()} to {test["Date"].max().date()}')

In [ ]:
# Generate all predictions
result_proba  = model_result.predict_proba(X_test)   # shape (n, 3): [away, draw, home]
result_pred   = model_result.predict(X_test)

btts_proba    = model_btts.predict_proba(X_test)[:, 1]
over25_proba  = model_over25.predict_proba(X_test)[:, 1]
over15_proba  = model_over15.predict_proba(X_test)[:, 1]
goals_pred    = model_goals.predict(X_test)

print('Predictions generated')

## 3. Brier Score

Measures how well-calibrated the probability outputs are.
- 0.00 = perfect
- 0.25 = no skill (equivalent to always predicting 0.5)
- Below 0.20 = well calibrated for football prediction

In [ ]:
# Result model — one Brier score per class
RESULT_LABELS = {0: 'Away Win', 1: 'Draw', 2: 'Home Win'}

print('=== Result Model (per class) ===')
result_brier_scores = {}
for i, label in RESULT_LABELS.items():
    bs = brier_score_loss(
        (test['result_encoded'] == i).astype(int),
        result_proba[:, i]
    )
    result_brier_scores[label] = bs
    print(f'  {label:12s}: {bs:.4f}')

print(f'  Mean       : {np.mean(list(result_brier_scores.values())):.4f}')

print()
print('=== Binary Models ===')
binary_models = [
    ('BTTS',     btts_proba,   test['btts']),
    ('Over 2.5', over25_proba, test['over_2_5']),
    ('Over 1.5', over15_proba, test['over_1_5']),
]
for name, proba, actual in binary_models:
    bs = brier_score_loss(actual, proba)
    print(f'  {name:10s}: {bs:.4f}')

## 4. Calibration Curves

A perfectly calibrated model lies on the diagonal.
- Curve above diagonal = model is underconfident (predicts lower probabilities than it should)
- Curve below diagonal = model is overconfident

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Calibration Curves — Soca Scores Models', fontsize=14, fontweight='bold')

calibration_targets = [
    ('BTTS',     btts_proba,   test['btts'],      axes[0, 0]),
    ('Over 2.5', over25_proba, test['over_2_5'],  axes[0, 1]),
    ('Over 1.5', over15_proba, test['over_1_5'],  axes[1, 0]),
    ('Result — Home Win', result_proba[:, 2],
     (test['result_encoded'] == 2).astype(int),   axes[1, 1]),
]

for name, proba, actual, ax in calibration_targets:
    fraction_pos, mean_pred = calibration_curve(actual, proba, n_bins=10, strategy='uniform')
    ax.plot(mean_pred, fraction_pos, marker='o', label='Model', color='steelblue')
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect calibration')
    ax.set_title(name)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('calibration_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: calibration_curves.png')

## 5. Confusion Matrix — Result Model

Shows where the model confuses Home Win / Draw / Away Win.
The draw class is almost always the hardest to predict in football.

In [ ]:
cm = confusion_matrix(test['result_encoded'], result_pred)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Away Win', 'Draw', 'Home Win']
)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Match Result Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Class breakdown
print('Actual class distribution in test set:')
for i, label in RESULT_LABELS.items():
    count = (test['result_encoded'] == i).sum()
    pct   = count / len(test) * 100
    print(f'  {label:12s}: {count:4d} ({pct:.1f}%)')

## 6. Backtesting Over Time

Tracks rolling accuracy across the test window.
Reveals whether the model degrades at certain points in the season or over time.

In [ ]:
backtest = test[['Date', 'result_encoded']].copy()
backtest['predicted'] = result_pred
backtest['correct']   = (backtest['predicted'] == backtest['result_encoded']).astype(int)
backtest = backtest.sort_values('Date').reset_index(drop=True)

# Rolling accuracy (window = 38 matches ~ 1 season)
backtest['rolling_acc'] = backtest['correct'].rolling(window=38, min_periods=10).mean()

# Monthly accuracy
backtest['YearMonth'] = backtest['Date'].dt.to_period('M')
monthly_acc = backtest.groupby('YearMonth')['correct'].mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9))
fig.suptitle('Backtesting — Match Result Model', fontsize=14, fontweight='bold')

# Rolling accuracy
ax1.plot(backtest['Date'], backtest['rolling_acc'], color='steelblue', linewidth=2)
ax1.axhline(y=backtest['correct'].mean(), color='red', linestyle='--',
            label=f'Overall accuracy: {backtest["correct"].mean():.3f}')
ax1.axhline(y=0.45, color='orange', linestyle=':', label='Home win baseline (~45%)')
ax1.set_title('Rolling Accuracy (38-match window)')
ax1.set_ylabel('Accuracy')
ax1.set_ylim(0.2, 0.8)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Monthly accuracy
monthly_acc.plot(kind='bar', ax=ax2, color='steelblue', alpha=0.8)
ax2.axhline(y=backtest['correct'].mean(), color='red', linestyle='--')
ax2.set_title('Monthly Accuracy')
ax2.set_ylabel('Accuracy')
ax2.set_xticklabels([str(p) for p in monthly_acc.index], rotation=45, ha='right')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('backtesting.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Baseline Comparison

The model must beat the simplest possible predictions to be useful.

- **Baseline 1**: Always predict Home Win (~45% of EPL matches are home wins)
- **Baseline 2**: Always predict the most common class in the training set
- **Baseline 3**: Random prediction weighted by class frequency

In [ ]:
y_true = test['result_encoded'].values

# Baseline 1 — always predict Home Win (class 2)
baseline_home     = np.full(len(y_true), 2)
baseline_home_acc = accuracy_score(y_true, baseline_home)

# Baseline 2 — most common class in test set
most_common       = pd.Series(y_true).value_counts().idxmax()
baseline_common   = np.full(len(y_true), most_common)
baseline_common_acc = accuracy_score(y_true, baseline_common)

# Baseline 3 — weighted random
class_weights = pd.Series(y_true).value_counts(normalize=True).sort_index().values
np.random.seed(42)
baseline_random   = np.random.choice([0, 1, 2], size=len(y_true), p=class_weights)
baseline_random_acc = accuracy_score(y_true, baseline_random)

# Model accuracy
model_acc = accuracy_score(y_true, result_pred)

print('=== Result Model Baseline Comparison ===')
print(f'  Model accuracy         : {model_acc:.4f}')
print(f'  Baseline: always home  : {baseline_home_acc:.4f}')
print(f'  Baseline: most common  : {baseline_common_acc:.4f}')
print(f'  Baseline: weighted rand: {baseline_random_acc:.4f}')
print()
print(f'  Model beats always-home by: {(model_acc - baseline_home_acc)*100:+.2f}%')

# Goals baseline — predict the mean total goals
mean_goals_pred = np.full(len(test), test['total_goals'].mean())
baseline_goals_mae = mean_absolute_error(test['total_goals'], mean_goals_pred)
model_goals_mae    = mean_absolute_error(test['total_goals'], goals_pred)

print()
print('=== Goals Model Baseline Comparison ===')
print(f'  Model MAE              : {model_goals_mae:.4f}')
print(f'  Baseline: predict mean : {baseline_goals_mae:.4f}')
print(f'  Improvement over mean  : {baseline_goals_mae - model_goals_mae:.4f}')

In [ ]:
# Summary bar chart
labels   = ['Model', 'Always\nHome Win', 'Most\nCommon', 'Weighted\nRandom']
accs     = [model_acc, baseline_home_acc, baseline_common_acc, baseline_random_acc]
colors   = ['steelblue', 'lightcoral', 'lightcoral', 'lightcoral']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, accs, color=colors, edgecolor='white', width=0.5)
ax.set_ylim(0, 0.7)
ax.set_ylabel('Accuracy')
ax.set_title('Model vs Baselines — Match Result Accuracy', fontsize=13, fontweight='bold')
ax.axhline(y=0.5267, color='green', linestyle='--', linewidth=1.2,
           label='2017 Challenge best (52.43%)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print('=' * 50)
print('EVALUATION SUMMARY')
print('=' * 50)
print(f'Test set size : {len(test)} matches')
print(f'Date range    : {test["Date"].min().date()} to {test["Date"].max().date()}')
print()
print('--- Accuracy ---')
print(f'Result        : {model_acc:.4f}')
print(f'BTTS          : {accuracy_score(test["btts"], (btts_proba > 0.5).astype(int)):.4f}')
print(f'Over 2.5      : {accuracy_score(test["over_2_5"], (over25_proba > 0.5).astype(int)):.4f}')
print(f'Over 1.5      : {accuracy_score(test["over_1_5"], (over15_proba > 0.5).astype(int)):.4f}')
print(f'Goals MAE     : {model_goals_mae:.4f}')
print()
print('--- Brier Score (lower = better, 0.25 = no skill) ---')
for label, bs in result_brier_scores.items():
    print(f'Result {label:10s}: {bs:.4f}')
print(f'BTTS          : {brier_score_loss(test["btts"], btts_proba):.4f}')
print(f'Over 2.5      : {brier_score_loss(test["over_2_5"], over25_proba):.4f}')
print(f'Over 1.5      : {brier_score_loss(test["over_1_5"], over15_proba):.4f}')
print()
print('--- vs Baselines ---')
print(f'Model vs always-home: {(model_acc - baseline_home_acc)*100:+.2f}%')
print(f'Goals vs mean pred  : {baseline_goals_mae - model_goals_mae:+.4f} MAE improvement')